In [ ]:
!pip install evaluate
!pip install bert-score
!pip install "transformers>=4.24.0,<5.0.0" --upgrade
!pip install IndicTransToolkit
import os
import random
import time
import copy
import re
import warnings
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import evaluate
from bert_score import score as bertscore
from tqdm.auto import tqdm
from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
)
from IndicTransToolkit import IndicProcessor
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

In [ ]:
# MOUNT GOOGLE DRIVE & FORCE STRICT LOCAL OFFLINE EXECUTION
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Absolute environment blocks to ensure zero network polling
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_EVALUATE_OFFLINE"] = "1"

# Local folder locations on Google Drive
MODEL_CHECKPOINT = "/content/drive/MyDrive/indictrans2_model"
ROBERTA_LOCAL_PATH = "/content/drive/MyDrive/indictrans2_model/roberta-large"

# Dataset file locations
TRAIN_SA = "drive/MyDrive/Data_to_Students/train_sa_10000.csv"
TRAIN_EN = "drive/MyDrive/Data_to_Students/train_en_10000.csv"
DEV_SA = "drive/MyDrive/Data_to_Students/dev_sa_1000.csv"
DEV_EN = "drive/MyDrive/Data_to_Students/dev_en_1000.csv"
TEST_SA = "drive/MyDrive/Data_to_Students/test_sa_1000.csv"

OUTPUT_DIR = "best_model"

SRC_MAX_LEN = 128
TGT_MAX_LEN = 128

BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01

BEAM_WIDTH = 5
GRADIENT_ACCUMULATION_STEPS = 2
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 50)
print("Device :", device)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA Version :", torch.version.cuda)
print("=" * 50)

bleu = evaluate.load("bleu")
print("BLEU metric initialized locally.")


In [ ]:
# DATA FRAME PARSING AND ALIGNMENT
train_sa = pd.read_csv(TRAIN_SA)
train_en = pd.read_csv(TRAIN_EN)
dev_sa = pd.read_csv(DEV_SA)
dev_en = pd.read_csv(DEV_EN)
test_sa = pd.read_csv(TEST_SA)

assert train_sa["Source_id"].equals(train_en["Source_id"]), "Training Source IDs do not match."
assert dev_sa["Source_id"].equals(dev_en["Source_id"]), "Development Source IDs do not match."

training_frame = train_sa.merge(train_en, on="Source_id", how="inner")
validation_frame = dev_sa.merge(dev_en, on="Source_id", how="inner")
testing_frame = test_sa.copy()

training_frame.rename(columns={"Sentence_sa": "source", "Sentence_en": "target"}, inplace=True)
validation_frame.rename(columns={"Sentence_sa": "source", "Sentence_en": "target"}, inplace=True)
testing_frame.rename(columns={"Sentence_sa": "source"}, inplace=True)

training_frame.dropna(inplace=True)
validation_frame.dropna(inplace=True)
testing_frame.dropna(inplace=True)

training_frame.reset_index(drop=True, inplace=True)
validation_frame.reset_index(drop=True, inplace=True)
testing_frame.reset_index(drop=True, inplace=True)

training_frame.drop_duplicates(subset=["source", "target"], inplace=True)
training_frame.reset_index(drop=True, inplace=True)

training_frame["src_len"] = training_frame["source"].astype(str).str.split().str.len()
training_frame["tgt_len"] = training_frame["target"].astype(str).str.split().str.len()


In [ ]:
# LOCAL TOKENIZER LOAD & TOKENIZATION
print("Initializing pipeline tokenizer directly from local Google Drive folder...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    trust_remote_code=True,
    local_files_only=True  
)
print("Tokenizer loaded successfully.")

SRC_LANG = "san_Deva"
TGT_LANG = "eng_Latn"
ip = IndicProcessor(inference=False)

def preprocess_batch(source_texts, target_texts=None):
    source_texts = ip.preprocess_batch(
        batch=source_texts,
        src_lang="san_Deva",
        tgt_lang="eng_Latn"
    )
    model_inputs = tokenizer(
        source_texts,
        max_length=SRC_MAX_LEN,
        truncation=True,
        padding="max_length"
    )
    if target_texts is not None:
        target_texts = ip.preprocess_batch(
            batch=target_texts,
            src_lang="eng_Latn",
            is_target=True
        )
        labels = tokenizer(
            text_target=target_texts,
            max_length=TGT_MAX_LEN,
            truncation=True,
            padding="max_length"
        )
        model_inputs["labels"] = labels["input_ids"]
    return model_inputs

class TranslationDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        self.encodings = preprocess_batch(
            self.df["source"].tolist(),
            self.df["target"].tolist()
        )
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        labels = torch.tensor(self.encodings["labels"][idx], dtype=torch.long)
        labels[labels == tokenizer.pad_token_id] = -100
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
            "labels": labels
        }

training_dataset = TranslationDataset(training_frame)
validation_dataset = TranslationDataset(validation_frame)

train_data_loader = DataLoader(training_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
dev_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)


In [ ]:
# LOCAL MODEL WEIGHTS LOADING
print("Loading translation model weights directly from local Google Drive folder...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_CHECKPOINT,
    trust_remote_code=True,
    local_files_only=True  
)
model.to(device)
print("Model configuration and network layers initialized successfully.")

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters      : {total:,}")
    print(f"Trainable Parameters  : {trainable:,}")
    return total, trainable

total_params, trainable_params = count_parameters(model)

no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)

total_training_steps = (math.ceil(len(train_data_loader) / GRADIENT_ACCUMULATION_STEPS) * EPOCHS)
USE_FP16 = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)


In [ ]:
# MODEL TRAINING CIRCUITS
def train_one_epoch(model, dataloader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            outputs = model(**batch)
            loss = outputs.loss
            loss = loss / GRADIENT_ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(dataloader)):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        progress_bar.set_postfix(loss=f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}")
    return total_loss / len(dataloader)

@torch.no_grad()
def validate(model, dataloader):
    model.eval()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Validation", leave=False)
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()
    return total_loss / len(dataloader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_training_steps
)

best_val_loss = float("inf")
train_losses = []
val_losses = []
patience = 3
early_stop_counter = 0
best_weights = None

for epoch in range(EPOCHS):
    print("=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("=" * 60)
    train_loss = train_one_epoch(model, train_data_loader, optimizer, scheduler, scaler)
    val_loss = validate(model, dev_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Valid Loss : {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        print("Saving best local model checkpoints...")
        best_weights = copy.deepcopy(model.state_dict())
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
    else:
        early_stop_counter += 1
        print(f"No improvement ({early_stop_counter}/{patience})")
        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break

if best_weights is not None:
    model.load_state_dict(best_weights)
print("Best local weights restored back to memory.")


In [ ]:
# LOCAL GENERATION CIRCUITS
def clean_prediction(text):
    text = re.sub(r"</?s>", "", text)
    text = text.replace("<pad>", "")
    text = text.replace("eng_Latn", "")
    text = text.replace("san_Deva", "")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return text.strip()

@torch.no_grad()
def translate_batch(texts, batch_size=16):
    model.eval()
    generated_texts = []
    ip_inference = IndicProcessor(inference=True)

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        processed = ip_inference.preprocess_batch(
            batch=batch,
            src_lang="san_Deva",
            tgt_lang="eng_Latn"
        )
        inputs = tokenizer(
            processed,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=SRC_MAX_LEN
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model.generate(
             **inputs,
             num_beams=BEAM_WIDTH,
             max_new_tokens=64,
             length_penalty=0.8,
             no_repeat_ngram_size=3,
             repetition_penalty=1.1,
             early_stopping=True,
             use_cache=False
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        decoded = [clean_prediction(text) for text in decoded]
        generated_texts.extend(decoded)
    return generated_texts


In [ ]:
start_time = time.time()
generated_texts = translate_batch(validation_frame["source"].tolist(), batch_size=16)
reference_texts = validation_frame["target"].tolist()
inference_time = time.time() - start_time
print(f"Inference Time: {inference_time:.2f} sec")

# BLEU Score Calculation
bleu.add_batch(predictions=generated_texts, references=[[r] for r in reference_texts])
bleu_metrics = bleu.compute()
print(f"BLEU Score : {bleu_metrics['bleu']:.4f}")

# BERTScore Calculation (Using local RoBERTa weights from Drive via monkey-patch)
print("Computing BERTScore using bert-base-uncased...")

BERT_MODEL = "bert-base-uncased"

bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL)
bert_model = BertModel.from_pretrained(BERT_MODEL)
bert_model.to(device)
bert_model.eval()

scorer = BERTScorer(
    model_type=BERT_MODEL,
    num_layers=12,
    device=str(device),
    batch_size=16,
    lang="en",
    rescale_with_baseline=True
)

P, R, F1 = scorer.score(generated_texts, reference_texts)

print(f"BERTScore Precision : {P.mean().item():.4f}")
print(f"BERTScore Recall    : {R.mean().item():.4f}")
print(f"BERTScore F1        : {F1.mean().item():.4f}")


# Final Inference and Submission Generation
start_time = time.time()
test_predictions = translate_batch(testing_frame["source"].tolist(), batch_size=32)
test_inference_time = time.time() - start_time
print(f"Test inference time : {test_inference_time:.2f} sec")

submission = pd.DataFrame({
    "Source_id": testing_frame["Source_id"],
    "Sentence_en": test_predictions
})

OUTPUT_PATH = "/content/drive/MyDrive/IndicTrans2_Output"

os.makedirs(OUTPUT_PATH, exist_ok=True)

submission.to_csv(
    f"{OUTPUT_PATH}/submission.csv",
    index=False,
    encoding="utf-8"
)

print(f"Saved to {OUTPUT_PATH}")


In [ ]:
# Translation Examples with Error Analysis

NUM_EXAMPLES = 10

examples = pd.DataFrame({
    "Source (Sanskrit)": validation_frame["source"].iloc[:NUM_EXAMPLES].values,
    "Reference (English)": reference_texts[:NUM_EXAMPLES],
    "Prediction (English)": generated_texts[:NUM_EXAMPLES]
})

def analyze_translation(reference, prediction):
    ref_words = set(reference.lower().split())
    pred_words = set(prediction.lower().split())

    common = len(ref_words & pred_words)
    precision = common / max(len(pred_words), 1)
    recall = common / max(len(ref_words), 1)

    if reference.strip().lower() == prediction.strip().lower():
        return "Exact match."

    if precision > 0.85 and recall > 0.85:
        return "Very close translation with minor lexical differences."

    elif precision > 0.65:
        return "Meaning largely preserved; some words/phrases differ."

    elif precision > 0.40:
        return "Partial translation. Missing or incorrect content."

    else:
        return "Poor semantic match; major translation errors."

examples["Error Analysis"] = [
    analyze_translation(ref, pred)
    for ref, pred in zip(
        examples["Reference (English)"],
        examples["Prediction (English)"]
    )
]

print("="*120)
print("Translation Examples")
print("="*120)

display(examples)